# 05 — Model Evaluation + Explainability (SHAP)
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Score the held-out test set via the WML endpoint
- Compute classification metrics (ROC-AUC, F1, Precision-Recall)
- Generate SHAP explanations — *why* a transaction was flagged as fraud
- Frame results in terms of LGPD compliance

**LGPD context:**  
> Art. 20 — O titular dos dados tem direito a solicitar revisão de decisões tomadas unicamente com base em tratamento automatizado.  
> SHAP values provide the per-feature contribution that enables this explanation.

In [ ]:
import sys
sys.path.append('..')

import json
import tarfile
import tempfile
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from src.wml_scorer import WMLScorer
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Test Set from DB2

In [ ]:
from src.db2_connector import DB2Connector

with DB2Connector('../config/credentials.json') as conn:
    test_df = conn.query("SELECT * FROM TRANSACTIONS WHERE SPLIT = 'test'")

test_df.columns = test_df.columns.str.lower()
y_test = test_df['fraude'].values
X_test = test_df.drop(columns=['fraude', 'split'])

print(f'Test set from DB2: {X_test.shape}')
print(f'Fraud rate: {y_test.mean():.4%}')

## 2. Score via WML Endpoint
Add `scoring_url` to your `config/credentials.json` from notebook 04.

In [ ]:
with open('../config/credentials.json') as f:
    config = json.load(f)

with open('../config/deployment_meta.json') as f:
    meta = json.load(f)

# WMLScorer lê scoring_url de deployment_meta.json — sem modificar credentials.json
config['wml']['scoring_url'] = meta['scoring_url']
scorer = WMLScorer('../config/credentials.json')

all_predictions = []
batch_size = 500
for i in range(0, len(X_test), batch_size):
    batch = X_test.iloc[i:i + batch_size]
    preds = scorer.score_dataframe(batch)
    all_predictions.extend(preds)
    if i % 5000 == 0:
        print(f'  Scored {min(i + batch_size, len(X_test)):,} / {len(X_test):,}')

y_pred = np.array([p['prediction'] for p in all_predictions])
y_prob = np.array([p['probability'][1] for p in all_predictions])
print(f'\nScoring complete. Predictions shape: {y_pred.shape}')

## 3. Classification Metrics

In [ ]:
auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=['Legítima', 'Fraude']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], color='#0062ff')
axes[0].set_title(f'ROC Curve (AUC = {auc:.3f})', fontweight='bold')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], color='#0062ff')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=axes[2],
    display_labels=['Legítima', 'Fraude'],
    colorbar=False, cmap='Blues'
)
axes[2].set_title('Confusion Matrix', fontweight='bold')

plt.suptitle('PIX Fraud Detector — Model Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Explainability
SHAP (SHapley Additive exPlanations) attributes model predictions to individual features,  
enabling human-readable explanations required by LGPD Art. 20.

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials
import joblib

wml_credentials = Credentials(api_key=config['wml']['apikey'], url=config['wml']['url'])
client = APIClient(wml_credentials, space_id=config['wml']['space_id'])

with tempfile.TemporaryDirectory() as tmpdir:
    archive_path = os.path.join(tmpdir, 'model.tar.gz')
    client.repository.download(meta['model_uid'], filename=archive_path)

    # tarfile em vez de os.system('tar') — compatível com Windows
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(tmpdir)

    pkl_files = [f for f in os.listdir(tmpdir) if f.endswith('.pkl')]
    if not pkl_files:
        raise FileNotFoundError('No .pkl file found in model archive.')
    pipeline = joblib.load(os.path.join(tmpdir, pkl_files[0]))

print(f'Pipeline loaded: {type(pipeline).__name__}')
print(f'Steps: {[s[0] for s in pipeline.steps]}')

In [ ]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test), size=500, replace=False)
X_shap = X_test.iloc[sample_idx]

final_estimator = pipeline.steps[-1][1]
explainer = shap.TreeExplainer(final_estimator)

from sklearn.pipeline import Pipeline as SKPipeline
preprocessor = SKPipeline(pipeline.steps[:-1])
X_transformed = preprocessor.transform(X_shap)

shap_values = explainer.shap_values(X_transformed)
print(f'SHAP values computed for {len(X_shap)} samples.')

In [ ]:
# Summary plot — global feature importance
shap_fraud = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_fraud, X_transformed,
    feature_names=X_test.columns.tolist(),
    show=False, max_display=15
)
plt.title('SHAP Feature Importance — Fraud Class', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../assets/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual explanation — explain a single flagged transaction
fraud_indices = np.where(y_pred[sample_idx] == 1)[0]
if len(fraud_indices) > 0:
    idx = fraud_indices[0]
    print(f'Explaining transaction #{sample_idx[idx]} (predicted: FRAUD)')
    shap.force_plot(
        explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        shap_fraud[idx],
        X_transformed[idx],
        feature_names=X_test.columns.tolist(),
        matplotlib=True,
        show=False
    )
    plt.title('LGPD Explanation — Why this transaction was flagged', fontweight='bold')
    plt.savefig('../assets/shap_force_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No fraud predictions in sample — try a larger sample size.')

## 5. Results Summary
Print a clean summary for the README.

In [ ]:
results = {
    'ROC-AUC':    round(roc_auc_score(y_test, y_prob), 4),
    'F1 (fraud)': round(f1_score(y_test, y_pred), 4),
    'Precision':  round(precision_score(y_test, y_pred), 4),
    'Recall':     round(recall_score(y_test, y_pred), 4),
}

print('=' * 40)
print('  PIX Fraud Detector — Final Results')
print('=' * 40)
for k, v in results.items():
    print(f'  {k:<20} {v}')
print('=' * 40)

with open('../config/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved to config/results.json')